# 02 — Diagnóstico da Rotulagem Semiautomática

Implementa a Fase 18 do plano de elaboração e a Seção 4.3 do documento
mestre (`configs/labeling.yaml`): mede a concordância da rotulagem em
cascata contra os gold sets externos (TweetSentBR/RePro) e investiga
amostras atípicas/de baixa confiança por dois sinais independentes —
hipóteses interpretáveis via HypotheSAEs e erro de reconstrução de um
autoencoder.

**Limitação conhecida** (ver `src/main.py`, `_build_labeling_stage_kwargs`):
apenas o rotulador heurístico-lexical (`heuristica_lexica`) está
implementado neste projeto — `llm_zero_shot` e `modelo_referencia` de
`configs/labeling.yaml -> cascade.labelers` ainda não existem. Por isso, a
concordância *entre rotuladores* não é avaliável ainda (exigiria ao menos
dois); a validação de qualidade aqui é feita contra os gold sets externos.

**Pré-requisito**: a etapa `preprocessing` já deve ter sido executada
(`uv run python src/main.py --stage preprocessing`), populando
`paths.normalized_corpus_file`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np

from config.constants import CONFIG_FILE_NAMES
from config.paths import CONFIGS_DIR, load_project_paths
from data.loader import load_labeled_corpus, load_raw_tweet_dataset, read_dataset_file
from io_utils.yaml import read_yaml
from labeling.automatic import LexicalHeuristicLabeler, run_cascade_labeling
from labeling.confidence import (
    calculate_agreement_ratio,
    calculate_discordance_score,
    flag_low_confidence_samples,
)
from labeling.consensus import aggregate_by_weighted_majority_vote
from labeling.validation import evaluate_against_gold_set

paths = load_project_paths()
labeling_config = read_yaml(CONFIGS_DIR / CONFIG_FILE_NAMES["labeling"])
labelers = {"heuristica_lexica": LexicalHeuristicLabeler()}

## Diagnóstico de confiança sobre o corpus normalizado próprio

Roda a cascata (com o único rotulador disponível) sobre
`paths.normalized_corpus_file` e calcula concordância/discordância por
amostra — com um único rotulador, `agreement_ratio` é trivialmente 1.0
para toda amostra, então o sinal realmente útil aqui vem da confiança
bruta do rotulador (`confidence`), usada para sinalizar candidatos à
validação humana.

In [ ]:
normalized_corpus = read_dataset_file(paths.normalized_corpus_file)
cascade_results = run_cascade_labeling(normalized_corpus, labelers, text_column="text_normalized")
consensus = aggregate_by_weighted_majority_vote(cascade_results)
agreement = calculate_agreement_ratio(cascade_results)
discordance = calculate_discordance_score(cascade_results)
low_confidence_flags = flag_low_confidence_samples(
    discordance,
    low_confidence_threshold=labeling_config["confidence"]["low_confidence_threshold"],
    discordance_threshold=labeling_config["confidence"]["discordance_threshold"],
)
print(
    f"{low_confidence_flags['requires_human_validation'].sum()}/{low_confidence_flags.height} "
    "amostra(s) sinalizada(s) para validação humana."
)
low_confidence_flags.head()

## Validação contra os gold sets externos (TweetSentBR/RePro)

Reaplica a mesma cascata sobre os textos de cada gold set já rotulado por
humanos (`configs/labeling.yaml -> gold_sets`) e compara com o rótulo
original via Cohen's Kappa (`validation.metric`), contra o mínimo aceitável
configurado (`validation.minimum_agreement`).

In [ ]:
gold_set_files = {"tweetsentbr": paths.tweetsentbr_file, "repro": paths.repro_file}

for gold_set_name, gold_set_path in gold_set_files.items():
    gold_set_enabled = labeling_config["gold_sets"][gold_set_name]["enabled"]
    if not gold_set_enabled or not gold_set_path.exists():
        print(f"Gold set '{gold_set_name}' desabilitado ou ausente em '{gold_set_path}'; pulando.")
        continue

    gold_corpus = load_labeled_corpus(gold_set_path)
    gold_cascade_results = run_cascade_labeling(gold_corpus, labelers, text_column="text")
    gold_predicted = aggregate_by_weighted_majority_vote(gold_cascade_results)

    validation_result = evaluate_against_gold_set(
        gold_predicted,
        gold_corpus,
        minimum_kappa=labeling_config["validation"]["minimum_agreement"],
    )
    print(
        f"{gold_set_name}: kappa={validation_result.cohen_kappa:.3f} "
        f"(n={validation_result.n_samples}, "
        f"atende mínimo={validation_result.meets_minimum_agreement})"
    )

## Hipóteses interpretáveis via HypotheSAEs

Segundo sinal de diagnóstico (`configs/labeling.yaml ->
diagnostics.hypothesaes`): embute as amostras sinalizadas para validação
humana e treina um autoencoder esparso para encontrar neurônios cuja
ativação prediz a própria sinalização de baixa confiança — cada neurônio
selecionado é interpretado em linguagem natural como uma hipótese sobre o
que torna essas amostras ambíguas.

In [ ]:
from features.contextual_embeddings import extract_contextual_embeddings, load_contextual_encoder
from hypothesaes.quickstart import generate_hypotheses, train_sae

diagnostic_sample = normalized_corpus.join(low_confidence_flags, on="id")
contextual_encoder = load_contextual_encoder()
sample_embeddings = extract_contextual_embeddings(
    diagnostic_sample, contextual_encoder, text_column="text_normalized"
)
embedding_columns = [column for column in sample_embeddings.columns if column != "id"]
embedding_matrix = sample_embeddings.select(embedding_columns).to_numpy()

sparse_autoencoder = train_sae(
    embedding_matrix,
    m_total_neurons=256,
    k_active_neurons=16,
    checkpoint_dir=paths.models_checkpoints_dir,
)
hypotheses = generate_hypotheses(
    diagnostic_sample["text_normalized"].to_list(),
    diagnostic_sample["requires_human_validation"].cast(int).to_list(),
    embedding_matrix,
    sparse_autoencoder,
    classification=True,
    n_selected_neurons=labeling_config["diagnostics"]["hypothesaes"]["n_hypotheses"],
)
hypotheses.head(10)

## Erro de reconstrução do autoencoder

Terceiro sinal, independente do HypotheSAEs: amostras com erro de
reconstrução acima do percentil configurado
(`diagnostics.autoencoder_reconstruction_error.anomaly_percentile`) são
atípicas em relação ao restante do corpus — comparadas aqui com as
amostras já sinalizadas por baixa confiança, para checar se os dois sinais
apontam para as mesmas amostras.

In [ ]:
from features.reduction import compute_reconstruction_error, train_autoencoder

autoencoder_artifacts = train_autoencoder(embedding_matrix, input_dim=embedding_matrix.shape[1])
reconstruction_errors = compute_reconstruction_error(embedding_matrix, autoencoder_artifacts)
anomaly_percentile = labeling_config["diagnostics"]["autoencoder_reconstruction_error"][
    "anomaly_percentile"
]
anomaly_threshold = np.percentile(reconstruction_errors, anomaly_percentile)
is_reconstruction_anomaly = reconstruction_errors >= anomaly_threshold

overlap_ratio = np.mean(
    is_reconstruction_anomaly & diagnostic_sample["requires_human_validation"].to_numpy()
)
print(
    f"{is_reconstruction_anomaly.sum()}/{len(reconstruction_errors)} amostra(s) acima do "
    f"percentil {anomaly_percentile} de erro de reconstrução; "
    f"sobreposição com baixa confiança: {overlap_ratio:.1%}"
)

## Conclusões

Registrar aqui: (1) se o kappa contra TweetSentBR/RePro atinge o mínimo
configurado (0.6) — caso não atinja, é um forte indício de que o
rotulador heurístico-lexical sozinho é insuficiente e reforça a
prioridade de implementar `llm_zero_shot`/`modelo_referencia`; (2) as
hipóteses do HypotheSAEs que mais se destacaram por fidelidade
(`{scoring_metric}_fidelity_score`); (3) o grau de sobreposição entre os
dois sinais de anomalia (baixa confiança vs. erro de reconstrução) — alta
sobreposição reforça a confiança nas amostras selecionadas para validação
humana (`configs/labeling.yaml -> human_validation`).